# Градиентный бустинг с learning rate: 20 шагов, каждый — отдельно

Тот же пример с четырьмя ноутбуками, что в Excel и на веб-странице урока. Отличие этого ноутбука: **ни одного цикла по шагам**.
Каждый шаг $m = 1,\dots,20$ развёрнут в собственные ячейки — пояснение, четыре этапа с таблицами-доказательствами и график шага, —
чтобы любую строку можно было выполнить, изменить и проверить руками.

| Этап | Что считаем | Формула | Основание |
|---|---|---|---|
| **A** | остатки | $r_i = y_i - F_{m-1}(x_i)$ | остаток = $-\partial L/\partial F$ при $L=\tfrac12(y-F)^2$ — антиградиент потери |
| **B** | поправки групп | $\gamma_G = \frac{1}{\lvert G\rvert}\sum_{i\in G} r_i$ | среднее минимизирует $\sum_{i\in G}(r_i-\gamma)^2$ |
| **C** | выбор колонки | $j^*=\arg\min_j \sum_{G\in j}\sum_{i\in G}(r_i-\gamma_G)^2$ | берём колонку, после которой ошибки остаётся меньше всего |
| **D** | обновление | $F_m(x_i)=F_{m-1}(x_i)+\nu\,h_m(x_i)$ | $\nu<1$ — неполный шаг, регуляризация Фридмана |

Библиотеки: только `numpy`, `pandas`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
C_SCREEN, C_SSD, C_TRUE, C_PRED, C_GREY = "#2E75B6", "#E07B2A", "#595959", "#1B7F4B", "#B0B0B0"

names = ["A", "B", "C", "D"]
df = pd.DataFrame({"экран": ['13"', '13"', '15"', '15"'], "ssd": ["256", "512", "256", "512"],
                   "цена": [820, 880, 1120, 1180]}, index=pd.Index(names, name="ноутбук"))
y = df["цена"].to_numpy(dtype=float)
FEATURES = {"экран": df["экран"].to_numpy(), "ssd": df["ssd"].to_numpy()}
LR = 0.5          # learning rate ν
df

## Вспомогательные функции

Каждая функция — ровно один этап. Они не считают «весь бустинг», а только оформляют одно действие, которое ниже вызывается на каждом шаге явно.

In [ ]:
def stage_A(F_prev, m):
    """A. Остатки: r = y − F_{m−1}. Возвращает r и таблицу."""
    r = y - F_prev
    t = pd.DataFrame({f"F_{m-1}(x)": F_prev, "y": y, "r = y − F": r, "|r|": np.abs(r)}, index=df.index)
    return r, t.round(4)

def stage_B(r):
    """B. Для каждой колонки: группы, сумма остатков, γ_G = среднее, остаток ошибки в группе."""
    rows, result = [], {}
    for col, vals in FEATURES.items():
        corr = np.zeros(4); groups = {}
        for g in dict.fromkeys(vals):
            idx = np.where(vals == g)[0]; gamma = r[idx].mean(); corr[idx] = gamma
            sse = ((r[idx] - gamma) ** 2).sum()
            groups[g] = gamma
            rows.append({"колонка": col, "группа": g, "состав": ", ".join(names[i] for i in idx),
                         "Σ r": r[idx].sum(), "|G|": len(idx), "γ_G = Σr/|G|": gamma, "остаётся Σ(r−γ)²": sse})
        result[col] = dict(corr=corr, groups=groups, sse=((r - corr) ** 2).sum())
    return result, pd.DataFrame(rows).set_index(["колонка", "группа"]).round(4)

def stage_C(cands):
    """C. Выбор колонки с наименьшей оставшейся суммой квадратов."""
    best = min(cands, key=lambda k: cands[k]["sse"])
    t = pd.DataFrame({"остаётся Σ(r−γ)²": {k: v["sse"] for k, v in cands.items()}}).round(4)
    t["выбор"] = ["◀ выбрана" if k == best else "" for k in t.index]
    return best, t

def stage_D(F_prev, h, lr, m, best):
    """D. F_m = F_{m−1} + ν·h_m, новые остатки."""
    F_new = F_prev + lr * h
    r_new = y - F_new
    t = pd.DataFrame({f"F_{m-1}(x)": F_prev, f"h_{m} по «{best}»": h, f"ν·h_{m}": lr * h, f"F_{m}(x)": F_new,
                      "y": y, "новый r": r_new, "|r|": np.abs(r_new), "r²": r_new ** 2}, index=df.index)
    return F_new, r_new, t.round(4)

def plot_step(m, F_prev, F_new, r_prev, r_new, best, cands):
    """График шага: остатки до/после, предсказания до/после против реальных, сумма квадратов кандидатов."""
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.4))
    x = np.arange(4); w = 0.36
    a1.bar(x - w/2, r_prev, w, color=C_GREY, label=f"до шага (Σ|r|={np.abs(r_prev).sum():g})")
    a1.bar(x + w/2, r_new, w, color=C_PRED, label=f"после (Σ|r|={np.abs(r_new).sum():g})")
    a1.axhline(0, color=C_TRUE, lw=1); a1.set_xticks(x, names); a1.set_title(f"Шаг {m}: остатки rᵢ до и после")
    a1.legend(fontsize=8)
    a2.scatter(x, y, color="black", zorder=5, label="реальная y")
    a2.plot(x, F_prev, "o--", color=C_GREY, label=f"F_{m-1}")
    a2.plot(x, F_new, "o-", color=C_SCREEN if best == "экран" else C_SSD, label=f"F_{m} (+ν·h по «{best}»)")
    for i in range(4):
        a2.annotate("", (x[i], F_new[i]), (x[i], F_prev[i]), arrowprops=dict(arrowstyle="->", color=C_PRED, lw=1.2))
    a2.set_xticks(x, names); a2.set_title("Предсказания: куда сдвинулись"); a2.legend(fontsize=8)
    cols = list(cands); vals = [cands[c]["sse"] for c in cols]
    a3.bar(cols, vals, color=[C_SCREEN if c == "экран" else C_SSD for c in cols])
    for i, v in enumerate(vals):
        a3.text(i, v, f"{v:g}", ha="center", va="bottom", fontsize=9)
    a3.set_title("Этап C: остаток ошибки по кандидатам"); a3.set_ylabel("Σ(r−γ)²")
    if best in cols:
        a3.get_xticklabels()[cols.index(best)].set_fontweight("bold")
    plt.tight_layout(); plt.show()

## Шаг 0 — стартовое предсказание $F_0$

Пока признаков нет, модель — одно число $c$. Минимизируем $S(c)=\sum_i (y_i-c)^2$:
$\frac{dS}{dc}=-2\sum_i (y_i-c)=0 \Rightarrow c=\bar y$. Среднее — не договорённость, а решение задачи минимизации.

In [ ]:
F0 = np.full(4, y.mean())
print(f"F0 = ({' + '.join(f'{v:g}' for v in y)}) / 4 = {F0[0]:g}")
r0 = y - F0
display(pd.DataFrame({"F_0": F0, "y": y, "r_0 = y − F_0": r0, "|r_0|": np.abs(r0)}, index=df.index))
print(f"Σ|r| = {np.abs(r0).sum():g},  Σr² = {(r0**2).sum():g}")

cs = np.linspace(800, 1200, 401); S = [((y - c) ** 2).sum() for c in cs]
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(cs, S, color=C_PRED, lw=2); ax.axvline(F0[0], color=C_TRUE, ls="--", lw=1)
ax.set_xlabel("c — одно предсказание для всех"); ax.set_ylabel("S(c) = Σ(yᵢ − c)²"); ax.set_title("Минимум суммы квадратов — в среднем")
plt.show()

## Шаг 1:  $F_{1} = F_{0} + \nu\,h_{1}$,  $\nu = 0.5$

**Где мы.** После шага 0 сумма модулей ошибок $\sum|r_i| = 600$, сумма квадратов $\sum r_i^2 = 93600$.
На этом шаге модель выберет колонку **«экран»**.

### 1.A — Остатки текущей модели $F_{0}$

$r_i = y_i - F_{0}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 1000 = -180$  
$r_B = 880 - 1000 = -120$  
$r_C = 1120 - 1000 = 120$  
$r_D = 1180 - 1000 = 180$

Проверка: $\sum|r_i| = 180 + 120 + 120 + 180 = 600$.

In [ ]:
r1, tA = stage_A(F0, m=1)
display(tA)
print(f"Σ|r| = {np.abs(r1).sum():g},  Σr² = {(r1**2).sum():g}")

### 1.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-180) + (-120))/2 = -150$; остаётся $((-180) - (-150))^2 + ((-120) - (-150))^2 = 1800$
- **экран 15"** = {C, D}: $\gamma = (120 + 180)/2 = 150$; остаётся $(120 - 150)^2 + (180 - 150)^2 = 1800$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-180) + 120)/2 = -30$; остаётся $((-180) - (-30))^2 + (120 - (-30))^2 = 45000$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-120) + 180)/2 = 30$; остаётся $((-120) - 30)^2 + (180 - 30)^2 = 45000$

In [ ]:
cands1, tB = stage_B(r1)
display(tB)

### 1.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $1800 + 1800 = 3600$;
SSD: $45000 + 45000 = 90000$.

Минимум у **«экран»** — в 25 раза меньше. Значит $h_{1}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best1, tC = stage_C(cands1)
display(tC)
h1 = cands1[best1]["corr"]
print("h_1 =", dict(zip(names, h1.round(4).tolist())))

### 1.D — Обновление модели с $\nu = 0.5$

$F_{1}(x_i) = F_{0}(x_i) + \nu\,h_{1}(x_i)$ — прибавляем только долю поправки.

$F_{1}(A) = 1000 + 0.5\cdot(-150) = 925$, новый остаток $820 - 925 = -105$  
$F_{1}(B) = 1000 + 0.5\cdot(-150) = 925$, новый остаток $880 - 925 = -45$  
$F_{1}(C) = 1000 + 0.5\cdot150 = 1075$, новый остаток $1120 - 1075 = 45$  
$F_{1}(D) = 1000 + 0.5\cdot150 = 1075$, новый остаток $1180 - 1075 = 105$

**Итог шага 1:** $\sum|r_i| = 300$ (было 600), $\sum r_i^2 = 26100$ (было 93600).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 150 осталось 75
— на следующем шаге эта колонка выиграет, только если 75 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F1, r1_new, tD = stage_D(F0, h1, LR, m=1, best=best1)
display(tD)
print(f"Σ|r| = {np.abs(r1_new).sum():g} (было {np.abs(r1).sum():g}),  Σr² = {(r1_new**2).sum():g}")
plot_step(1, F0, F1, r1, r1_new, best1, cands1)

## Шаг 2:  $F_{2} = F_{1} + \nu\,h_{2}$,  $\nu = 0.5$

**Где мы.** После шага 1 сумма модулей ошибок $\sum|r_i| = 300$, сумма квадратов $\sum r_i^2 = 26100$.
На этом шаге модель снова выберет колонку **«экран»** — её невыученный вклад 75 больше вклада другой колонки.

### 2.A — Остатки текущей модели $F_{1}$

$r_i = y_i - F_{1}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 925 = -105$  
$r_B = 880 - 925 = -45$  
$r_C = 1120 - 1075 = 45$  
$r_D = 1180 - 1075 = 105$

Проверка: $\sum|r_i| = 105 + 45 + 45 + 105 = 300$.

In [ ]:
r2, tA = stage_A(F1, m=2)
display(tA)
print(f"Σ|r| = {np.abs(r2).sum():g},  Σr² = {(r2**2).sum():g}")

### 2.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-105) + (-45))/2 = -75$; остаётся $((-105) - (-75))^2 + ((-45) - (-75))^2 = 1800$
- **экран 15"** = {C, D}: $\gamma = (45 + 105)/2 = 75$; остаётся $(45 - 75)^2 + (105 - 75)^2 = 1800$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-105) + 45)/2 = -30$; остаётся $((-105) - (-30))^2 + (45 - (-30))^2 = 11250$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-45) + 105)/2 = 30$; остаётся $((-45) - 30)^2 + (105 - 30)^2 = 11250$

In [ ]:
cands2, tB = stage_B(r2)
display(tB)

### 2.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $1800 + 1800 = 3600$;
SSD: $11250 + 11250 = 22500$.

Минимум у **«экран»** — в 6.25 раза меньше. Значит $h_{2}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best2, tC = stage_C(cands2)
display(tC)
h2 = cands2[best2]["corr"]
print("h_2 =", dict(zip(names, h2.round(4).tolist())))

### 2.D — Обновление модели с $\nu = 0.5$

$F_{2}(x_i) = F_{1}(x_i) + \nu\,h_{2}(x_i)$ — прибавляем только долю поправки.

$F_{2}(A) = 925 + 0.5\cdot(-75) = 887.5$, новый остаток $820 - 887.5 = -67.5$  
$F_{2}(B) = 925 + 0.5\cdot(-75) = 887.5$, новый остаток $880 - 887.5 = -7.5$  
$F_{2}(C) = 1075 + 0.5\cdot75 = 1112.5$, новый остаток $1120 - 1112.5 = 7.5$  
$F_{2}(D) = 1075 + 0.5\cdot75 = 1112.5$, новый остаток $1180 - 1112.5 = 67.5$

**Итог шага 2:** $\sum|r_i| = 150$ (было 300), $\sum r_i^2 = 9225$ (было 26100).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 75 осталось 37.5
— на следующем шаге эта колонка выиграет, только если 37.5 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F2, r2_new, tD = stage_D(F1, h2, LR, m=2, best=best2)
display(tD)
print(f"Σ|r| = {np.abs(r2_new).sum():g} (было {np.abs(r2).sum():g}),  Σr² = {(r2_new**2).sum():g}")
plot_step(2, F1, F2, r2, r2_new, best2, cands2)

## Шаг 3:  $F_{3} = F_{2} + \nu\,h_{3}$,  $\nu = 0.5$

**Где мы.** После шага 2 сумма модулей ошибок $\sum|r_i| = 150$, сумма квадратов $\sum r_i^2 = 9225$.
На этом шаге модель снова выберет колонку **«экран»** — её невыученный вклад 37.5 больше вклада другой колонки.

### 3.A — Остатки текущей модели $F_{2}$

$r_i = y_i - F_{2}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 887.5 = -67.5$  
$r_B = 880 - 887.5 = -7.5$  
$r_C = 1120 - 1112.5 = 7.5$  
$r_D = 1180 - 1112.5 = 67.5$

Проверка: $\sum|r_i| = 67.5 + 7.5 + 7.5 + 67.5 = 150$.

In [ ]:
r3, tA = stage_A(F2, m=3)
display(tA)
print(f"Σ|r| = {np.abs(r3).sum():g},  Σr² = {(r3**2).sum():g}")

### 3.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-67.5) + (-7.5))/2 = -37.5$; остаётся $((-67.5) - (-37.5))^2 + ((-7.5) - (-37.5))^2 = 1800$
- **экран 15"** = {C, D}: $\gamma = (7.5 + 67.5)/2 = 37.5$; остаётся $(7.5 - 37.5)^2 + (67.5 - 37.5)^2 = 1800$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-67.5) + 7.5)/2 = -30$; остаётся $((-67.5) - (-30))^2 + (7.5 - (-30))^2 = 2812.5$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-7.5) + 67.5)/2 = 30$; остаётся $((-7.5) - 30)^2 + (67.5 - 30)^2 = 2812.5$

In [ ]:
cands3, tB = stage_B(r3)
display(tB)

### 3.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $1800 + 1800 = 3600$;
SSD: $2812.5 + 2812.5 = 5625$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{3}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best3, tC = stage_C(cands3)
display(tC)
h3 = cands3[best3]["corr"]
print("h_3 =", dict(zip(names, h3.round(4).tolist())))

### 3.D — Обновление модели с $\nu = 0.5$

$F_{3}(x_i) = F_{2}(x_i) + \nu\,h_{3}(x_i)$ — прибавляем только долю поправки.

$F_{3}(A) = 887.5 + 0.5\cdot(-37.5) = 868.75$, новый остаток $820 - 868.75 = -48.75$  
$F_{3}(B) = 887.5 + 0.5\cdot(-37.5) = 868.75$, новый остаток $880 - 868.75 = 11.25$  
$F_{3}(C) = 1112.5 + 0.5\cdot37.5 = 1131.25$, новый остаток $1120 - 1131.25 = -11.25$  
$F_{3}(D) = 1112.5 + 0.5\cdot37.5 = 1131.25$, новый остаток $1180 - 1131.25 = 48.75$

**Итог шага 3:** $\sum|r_i| = 120$ (было 150), $\sum r_i^2 = 5006.25$ (было 9225).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 37.5 осталось 18.75
— на следующем шаге эта колонка выиграет, только если 18.75 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F3, r3_new, tD = stage_D(F2, h3, LR, m=3, best=best3)
display(tD)
print(f"Σ|r| = {np.abs(r3_new).sum():g} (было {np.abs(r3).sum():g}),  Σr² = {(r3_new**2).sum():g}")
plot_step(3, F2, F3, r3, r3_new, best3, cands3)

## Шаг 4:  $F_{4} = F_{3} + \nu\,h_{4}$,  $\nu = 0.5$

**Где мы.** После шага 3 сумма модулей ошибок $\sum|r_i| = 120$, сумма квадратов $\sum r_i^2 = 5006.25$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 30 больше вклада другой колонки.

### 4.A — Остатки текущей модели $F_{3}$

$r_i = y_i - F_{3}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 868.75 = -48.75$  
$r_B = 880 - 868.75 = 11.25$  
$r_C = 1120 - 1131.25 = -11.25$  
$r_D = 1180 - 1131.25 = 48.75$

Проверка: $\sum|r_i| = 48.75 + 11.25 + 11.25 + 48.75 = 120$.

In [ ]:
r4, tA = stage_A(F3, m=4)
display(tA)
print(f"Σ|r| = {np.abs(r4).sum():g},  Σr² = {(r4**2).sum():g}")

### 4.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-48.75) + 11.25)/2 = -18.75$; остаётся $((-48.75) - (-18.75))^2 + (11.25 - (-18.75))^2 = 1800$
- **экран 15"** = {C, D}: $\gamma = ((-11.25) + 48.75)/2 = 18.75$; остаётся $((-11.25) - 18.75)^2 + (48.75 - 18.75)^2 = 1800$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-48.75) + (-11.25))/2 = -30$; остаётся $((-48.75) - (-30))^2 + ((-11.25) - (-30))^2 = 703.125$
- **ssd 512 ГБ** = {B, D}: $\gamma = (11.25 + 48.75)/2 = 30$; остаётся $(11.25 - 30)^2 + (48.75 - 30)^2 = 703.125$

In [ ]:
cands4, tB = stage_B(r4)
display(tB)

### 4.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $1800 + 1800 = 3600$;
SSD: $703.125 + 703.125 = 1406.25$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{4}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best4, tC = stage_C(cands4)
display(tC)
h4 = cands4[best4]["corr"]
print("h_4 =", dict(zip(names, h4.round(4).tolist())))

### 4.D — Обновление модели с $\nu = 0.5$

$F_{4}(x_i) = F_{3}(x_i) + \nu\,h_{4}(x_i)$ — прибавляем только долю поправки.

$F_{4}(A) = 868.75 + 0.5\cdot(-30) = 853.75$, новый остаток $820 - 853.75 = -33.75$  
$F_{4}(B) = 868.75 + 0.5\cdot30 = 883.75$, новый остаток $880 - 883.75 = -3.75$  
$F_{4}(C) = 1131.25 + 0.5\cdot(-30) = 1116.25$, новый остаток $1120 - 1116.25 = 3.75$  
$F_{4}(D) = 1131.25 + 0.5\cdot30 = 1146.25$, новый остаток $1180 - 1146.25 = 33.75$

**Итог шага 4:** $\sum|r_i| = 75$ (было 120), $\sum r_i^2 = 2306.25$ (было 5006.25).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 30 осталось 15
— на следующем шаге эта колонка выиграет, только если 15 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F4, r4_new, tD = stage_D(F3, h4, LR, m=4, best=best4)
display(tD)
print(f"Σ|r| = {np.abs(r4_new).sum():g} (было {np.abs(r4).sum():g}),  Σr² = {(r4_new**2).sum():g}")
plot_step(4, F3, F4, r4, r4_new, best4, cands4)

## Шаг 5:  $F_{5} = F_{4} + \nu\,h_{5}$,  $\nu = 0.5$

**Где мы.** После шага 4 сумма модулей ошибок $\sum|r_i| = 75$, сумма квадратов $\sum r_i^2 = 2306.25$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 18.75 больше вклада другой колонки.

### 5.A — Остатки текущей модели $F_{4}$

$r_i = y_i - F_{4}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 853.75 = -33.75$  
$r_B = 880 - 883.75 = -3.75$  
$r_C = 1120 - 1116.25 = 3.75$  
$r_D = 1180 - 1146.25 = 33.75$

Проверка: $\sum|r_i| = 33.75 + 3.75 + 3.75 + 33.75 = 75$.

In [ ]:
r5, tA = stage_A(F4, m=5)
display(tA)
print(f"Σ|r| = {np.abs(r5).sum():g},  Σr² = {(r5**2).sum():g}")

### 5.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-33.75) + (-3.75))/2 = -18.75$; остаётся $((-33.75) - (-18.75))^2 + ((-3.75) - (-18.75))^2 = 450$
- **экран 15"** = {C, D}: $\gamma = (3.75 + 33.75)/2 = 18.75$; остаётся $(3.75 - 18.75)^2 + (33.75 - 18.75)^2 = 450$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-33.75) + 3.75)/2 = -15$; остаётся $((-33.75) - (-15))^2 + (3.75 - (-15))^2 = 703.125$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-3.75) + 33.75)/2 = 15$; остаётся $((-3.75) - 15)^2 + (33.75 - 15)^2 = 703.125$

In [ ]:
cands5, tB = stage_B(r5)
display(tB)

### 5.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $450 + 450 = 900$;
SSD: $703.125 + 703.125 = 1406.25$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{5}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best5, tC = stage_C(cands5)
display(tC)
h5 = cands5[best5]["corr"]
print("h_5 =", dict(zip(names, h5.round(4).tolist())))

### 5.D — Обновление модели с $\nu = 0.5$

$F_{5}(x_i) = F_{4}(x_i) + \nu\,h_{5}(x_i)$ — прибавляем только долю поправки.

$F_{5}(A) = 853.75 + 0.5\cdot(-18.75) = 844.375$, новый остаток $820 - 844.375 = -24.375$  
$F_{5}(B) = 883.75 + 0.5\cdot(-18.75) = 874.375$, новый остаток $880 - 874.375 = 5.625$  
$F_{5}(C) = 1116.25 + 0.5\cdot18.75 = 1125.625$, новый остаток $1120 - 1125.625 = -5.625$  
$F_{5}(D) = 1146.25 + 0.5\cdot18.75 = 1155.625$, новый остаток $1180 - 1155.625 = 24.375$

**Итог шага 5:** $\sum|r_i| = 60$ (было 75), $\sum r_i^2 = 1251.5625$ (было 2306.25).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 18.75 осталось 9.375
— на следующем шаге эта колонка выиграет, только если 9.375 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F5, r5_new, tD = stage_D(F4, h5, LR, m=5, best=best5)
display(tD)
print(f"Σ|r| = {np.abs(r5_new).sum():g} (было {np.abs(r5).sum():g}),  Σr² = {(r5_new**2).sum():g}")
plot_step(5, F4, F5, r5, r5_new, best5, cands5)

## Шаг 6:  $F_{6} = F_{5} + \nu\,h_{6}$,  $\nu = 0.5$

**Где мы.** После шага 5 сумма модулей ошибок $\sum|r_i| = 60$, сумма квадратов $\sum r_i^2 = 1251.5625$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 15 больше вклада другой колонки.

### 6.A — Остатки текущей модели $F_{5}$

$r_i = y_i - F_{5}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 844.375 = -24.375$  
$r_B = 880 - 874.375 = 5.625$  
$r_C = 1120 - 1125.625 = -5.625$  
$r_D = 1180 - 1155.625 = 24.375$

Проверка: $\sum|r_i| = 24.375 + 5.625 + 5.625 + 24.375 = 60$.

In [ ]:
r6, tA = stage_A(F5, m=6)
display(tA)
print(f"Σ|r| = {np.abs(r6).sum():g},  Σr² = {(r6**2).sum():g}")

### 6.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-24.375) + 5.625)/2 = -9.375$; остаётся $((-24.375) - (-9.375))^2 + (5.625 - (-9.375))^2 = 450$
- **экран 15"** = {C, D}: $\gamma = ((-5.625) + 24.375)/2 = 9.375$; остаётся $((-5.625) - 9.375)^2 + (24.375 - 9.375)^2 = 450$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-24.375) + (-5.625))/2 = -15$; остаётся $((-24.375) - (-15))^2 + ((-5.625) - (-15))^2 = 175.7812$
- **ssd 512 ГБ** = {B, D}: $\gamma = (5.625 + 24.375)/2 = 15$; остаётся $(5.625 - 15)^2 + (24.375 - 15)^2 = 175.7812$

In [ ]:
cands6, tB = stage_B(r6)
display(tB)

### 6.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $450 + 450 = 900$;
SSD: $175.7812 + 175.7812 = 351.5625$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{6}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best6, tC = stage_C(cands6)
display(tC)
h6 = cands6[best6]["corr"]
print("h_6 =", dict(zip(names, h6.round(4).tolist())))

### 6.D — Обновление модели с $\nu = 0.5$

$F_{6}(x_i) = F_{5}(x_i) + \nu\,h_{6}(x_i)$ — прибавляем только долю поправки.

$F_{6}(A) = 844.375 + 0.5\cdot(-15) = 836.875$, новый остаток $820 - 836.875 = -16.875$  
$F_{6}(B) = 874.375 + 0.5\cdot15 = 881.875$, новый остаток $880 - 881.875 = -1.875$  
$F_{6}(C) = 1125.625 + 0.5\cdot(-15) = 1118.125$, новый остаток $1120 - 1118.125 = 1.875$  
$F_{6}(D) = 1155.625 + 0.5\cdot15 = 1163.125$, новый остаток $1180 - 1163.125 = 16.875$

**Итог шага 6:** $\sum|r_i| = 37.5$ (было 60), $\sum r_i^2 = 576.5625$ (было 1251.5625).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 15 осталось 7.5
— на следующем шаге эта колонка выиграет, только если 7.5 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F6, r6_new, tD = stage_D(F5, h6, LR, m=6, best=best6)
display(tD)
print(f"Σ|r| = {np.abs(r6_new).sum():g} (было {np.abs(r6).sum():g}),  Σr² = {(r6_new**2).sum():g}")
plot_step(6, F5, F6, r6, r6_new, best6, cands6)

## Шаг 7:  $F_{7} = F_{6} + \nu\,h_{7}$,  $\nu = 0.5$

**Где мы.** После шага 6 сумма модулей ошибок $\sum|r_i| = 37.5$, сумма квадратов $\sum r_i^2 = 576.5625$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 9.375 больше вклада другой колонки.

### 7.A — Остатки текущей модели $F_{6}$

$r_i = y_i - F_{6}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 836.875 = -16.875$  
$r_B = 880 - 881.875 = -1.875$  
$r_C = 1120 - 1118.125 = 1.875$  
$r_D = 1180 - 1163.125 = 16.875$

Проверка: $\sum|r_i| = 16.875 + 1.875 + 1.875 + 16.875 = 37.5$.

In [ ]:
r7, tA = stage_A(F6, m=7)
display(tA)
print(f"Σ|r| = {np.abs(r7).sum():g},  Σr² = {(r7**2).sum():g}")

### 7.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-16.875) + (-1.875))/2 = -9.375$; остаётся $((-16.875) - (-9.375))^2 + ((-1.875) - (-9.375))^2 = 112.5$
- **экран 15"** = {C, D}: $\gamma = (1.875 + 16.875)/2 = 9.375$; остаётся $(1.875 - 9.375)^2 + (16.875 - 9.375)^2 = 112.5$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-16.875) + 1.875)/2 = -7.5$; остаётся $((-16.875) - (-7.5))^2 + (1.875 - (-7.5))^2 = 175.7812$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-1.875) + 16.875)/2 = 7.5$; остаётся $((-1.875) - 7.5)^2 + (16.875 - 7.5)^2 = 175.7812$

In [ ]:
cands7, tB = stage_B(r7)
display(tB)

### 7.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $112.5 + 112.5 = 225$;
SSD: $175.7812 + 175.7812 = 351.5625$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{7}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best7, tC = stage_C(cands7)
display(tC)
h7 = cands7[best7]["corr"]
print("h_7 =", dict(zip(names, h7.round(4).tolist())))

### 7.D — Обновление модели с $\nu = 0.5$

$F_{7}(x_i) = F_{6}(x_i) + \nu\,h_{7}(x_i)$ — прибавляем только долю поправки.

$F_{7}(A) = 836.875 + 0.5\cdot(-9.375) = 832.1875$, новый остаток $820 - 832.1875 = -12.1875$  
$F_{7}(B) = 881.875 + 0.5\cdot(-9.375) = 877.1875$, новый остаток $880 - 877.1875 = 2.8125$  
$F_{7}(C) = 1118.125 + 0.5\cdot9.375 = 1122.8125$, новый остаток $1120 - 1122.8125 = -2.8125$  
$F_{7}(D) = 1163.125 + 0.5\cdot9.375 = 1167.8125$, новый остаток $1180 - 1167.8125 = 12.1875$

**Итог шага 7:** $\sum|r_i| = 30$ (было 37.5), $\sum r_i^2 = 312.8906$ (было 576.5625).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 9.375 осталось 4.6875
— на следующем шаге эта колонка выиграет, только если 4.6875 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F7, r7_new, tD = stage_D(F6, h7, LR, m=7, best=best7)
display(tD)
print(f"Σ|r| = {np.abs(r7_new).sum():g} (было {np.abs(r7).sum():g}),  Σr² = {(r7_new**2).sum():g}")
plot_step(7, F6, F7, r7, r7_new, best7, cands7)

## Шаг 8:  $F_{8} = F_{7} + \nu\,h_{8}$,  $\nu = 0.5$

**Где мы.** После шага 7 сумма модулей ошибок $\sum|r_i| = 30$, сумма квадратов $\sum r_i^2 = 312.8906$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 7.5 больше вклада другой колонки.

### 8.A — Остатки текущей модели $F_{7}$

$r_i = y_i - F_{7}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 832.1875 = -12.1875$  
$r_B = 880 - 877.1875 = 2.8125$  
$r_C = 1120 - 1122.8125 = -2.8125$  
$r_D = 1180 - 1167.8125 = 12.1875$

Проверка: $\sum|r_i| = 12.1875 + 2.8125 + 2.8125 + 12.1875 = 30$.

In [ ]:
r8, tA = stage_A(F7, m=8)
display(tA)
print(f"Σ|r| = {np.abs(r8).sum():g},  Σr² = {(r8**2).sum():g}")

### 8.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-12.1875) + 2.8125)/2 = -4.6875$; остаётся $((-12.1875) - (-4.6875))^2 + (2.8125 - (-4.6875))^2 = 112.5$
- **экран 15"** = {C, D}: $\gamma = ((-2.8125) + 12.1875)/2 = 4.6875$; остаётся $((-2.8125) - 4.6875)^2 + (12.1875 - 4.6875)^2 = 112.5$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-12.1875) + (-2.8125))/2 = -7.5$; остаётся $((-12.1875) - (-7.5))^2 + ((-2.8125) - (-7.5))^2 = 43.9453$
- **ssd 512 ГБ** = {B, D}: $\gamma = (2.8125 + 12.1875)/2 = 7.5$; остаётся $(2.8125 - 7.5)^2 + (12.1875 - 7.5)^2 = 43.9453$

In [ ]:
cands8, tB = stage_B(r8)
display(tB)

### 8.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $112.5 + 112.5 = 225$;
SSD: $43.9453 + 43.9453 = 87.8906$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{8}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best8, tC = stage_C(cands8)
display(tC)
h8 = cands8[best8]["corr"]
print("h_8 =", dict(zip(names, h8.round(4).tolist())))

### 8.D — Обновление модели с $\nu = 0.5$

$F_{8}(x_i) = F_{7}(x_i) + \nu\,h_{8}(x_i)$ — прибавляем только долю поправки.

$F_{8}(A) = 832.1875 + 0.5\cdot(-7.5) = 828.4375$, новый остаток $820 - 828.4375 = -8.4375$  
$F_{8}(B) = 877.1875 + 0.5\cdot7.5 = 880.9375$, новый остаток $880 - 880.9375 = -0.9375$  
$F_{8}(C) = 1122.8125 + 0.5\cdot(-7.5) = 1119.0625$, новый остаток $1120 - 1119.0625 = 0.9375$  
$F_{8}(D) = 1167.8125 + 0.5\cdot7.5 = 1171.5625$, новый остаток $1180 - 1171.5625 = 8.4375$

**Итог шага 8:** $\sum|r_i| = 18.75$ (было 30), $\sum r_i^2 = 144.1406$ (было 312.8906).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 7.5 осталось 3.75
— на следующем шаге эта колонка выиграет, только если 3.75 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F8, r8_new, tD = stage_D(F7, h8, LR, m=8, best=best8)
display(tD)
print(f"Σ|r| = {np.abs(r8_new).sum():g} (было {np.abs(r8).sum():g}),  Σr² = {(r8_new**2).sum():g}")
plot_step(8, F7, F8, r8, r8_new, best8, cands8)

## Шаг 9:  $F_{9} = F_{8} + \nu\,h_{9}$,  $\nu = 0.5$

**Где мы.** После шага 8 сумма модулей ошибок $\sum|r_i| = 18.75$, сумма квадратов $\sum r_i^2 = 144.1406$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 4.6875 больше вклада другой колонки.

### 9.A — Остатки текущей модели $F_{8}$

$r_i = y_i - F_{8}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 828.4375 = -8.4375$  
$r_B = 880 - 880.9375 = -0.9375$  
$r_C = 1120 - 1119.0625 = 0.9375$  
$r_D = 1180 - 1171.5625 = 8.4375$

Проверка: $\sum|r_i| = 8.4375 + 0.9375 + 0.9375 + 8.4375 = 18.75$.

In [ ]:
r9, tA = stage_A(F8, m=9)
display(tA)
print(f"Σ|r| = {np.abs(r9).sum():g},  Σr² = {(r9**2).sum():g}")

### 9.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-8.4375) + (-0.9375))/2 = -4.6875$; остаётся $((-8.4375) - (-4.6875))^2 + ((-0.9375) - (-4.6875))^2 = 28.125$
- **экран 15"** = {C, D}: $\gamma = (0.9375 + 8.4375)/2 = 4.6875$; остаётся $(0.9375 - 4.6875)^2 + (8.4375 - 4.6875)^2 = 28.125$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-8.4375) + 0.9375)/2 = -3.75$; остаётся $((-8.4375) - (-3.75))^2 + (0.9375 - (-3.75))^2 = 43.9453$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-0.9375) + 8.4375)/2 = 3.75$; остаётся $((-0.9375) - 3.75)^2 + (8.4375 - 3.75)^2 = 43.9453$

In [ ]:
cands9, tB = stage_B(r9)
display(tB)

### 9.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $28.125 + 28.125 = 56.25$;
SSD: $43.9453 + 43.9453 = 87.8906$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{9}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best9, tC = stage_C(cands9)
display(tC)
h9 = cands9[best9]["corr"]
print("h_9 =", dict(zip(names, h9.round(4).tolist())))

### 9.D — Обновление модели с $\nu = 0.5$

$F_{9}(x_i) = F_{8}(x_i) + \nu\,h_{9}(x_i)$ — прибавляем только долю поправки.

$F_{9}(A) = 828.4375 + 0.5\cdot(-4.6875) = 826.0938$, новый остаток $820 - 826.0938 = -6.0938$  
$F_{9}(B) = 880.9375 + 0.5\cdot(-4.6875) = 878.5938$, новый остаток $880 - 878.5938 = 1.4062$  
$F_{9}(C) = 1119.0625 + 0.5\cdot4.6875 = 1121.4062$, новый остаток $1120 - 1121.4062 = -1.4062$  
$F_{9}(D) = 1171.5625 + 0.5\cdot4.6875 = 1173.9062$, новый остаток $1180 - 1173.9062 = 6.0938$

**Итог шага 9:** $\sum|r_i| = 15$ (было 18.75), $\sum r_i^2 = 78.2227$ (было 144.1406).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 4.6875 осталось 2.3438
— на следующем шаге эта колонка выиграет, только если 2.3438 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F9, r9_new, tD = stage_D(F8, h9, LR, m=9, best=best9)
display(tD)
print(f"Σ|r| = {np.abs(r9_new).sum():g} (было {np.abs(r9).sum():g}),  Σr² = {(r9_new**2).sum():g}")
plot_step(9, F8, F9, r9, r9_new, best9, cands9)

## Шаг 10:  $F_{10} = F_{9} + \nu\,h_{10}$,  $\nu = 0.5$

**Где мы.** После шага 9 сумма модулей ошибок $\sum|r_i| = 15$, сумма квадратов $\sum r_i^2 = 78.2227$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 3.75 больше вклада другой колонки.

### 10.A — Остатки текущей модели $F_{9}$

$r_i = y_i - F_{9}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 826.0938 = -6.0938$  
$r_B = 880 - 878.5938 = 1.4062$  
$r_C = 1120 - 1121.4062 = -1.4062$  
$r_D = 1180 - 1173.9062 = 6.0938$

Проверка: $\sum|r_i| = 6.0938 + 1.4062 + 1.4062 + 6.0938 = 15$.

In [ ]:
r10, tA = stage_A(F9, m=10)
display(tA)
print(f"Σ|r| = {np.abs(r10).sum():g},  Σr² = {(r10**2).sum():g}")

### 10.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-6.0938) + 1.4062)/2 = -2.3438$; остаётся $((-6.0938) - (-2.3438))^2 + (1.4062 - (-2.3438))^2 = 28.125$
- **экран 15"** = {C, D}: $\gamma = ((-1.4062) + 6.0938)/2 = 2.3438$; остаётся $((-1.4062) - 2.3438)^2 + (6.0938 - 2.3438)^2 = 28.125$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-6.0938) + (-1.4062))/2 = -3.75$; остаётся $((-6.0938) - (-3.75))^2 + ((-1.4062) - (-3.75))^2 = 10.9863$
- **ssd 512 ГБ** = {B, D}: $\gamma = (1.4062 + 6.0938)/2 = 3.75$; остаётся $(1.4062 - 3.75)^2 + (6.0938 - 3.75)^2 = 10.9863$

In [ ]:
cands10, tB = stage_B(r10)
display(tB)

### 10.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $28.125 + 28.125 = 56.25$;
SSD: $10.9863 + 10.9863 = 21.9727$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{10}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best10, tC = stage_C(cands10)
display(tC)
h10 = cands10[best10]["corr"]
print("h_10 =", dict(zip(names, h10.round(4).tolist())))

### 10.D — Обновление модели с $\nu = 0.5$

$F_{10}(x_i) = F_{9}(x_i) + \nu\,h_{10}(x_i)$ — прибавляем только долю поправки.

$F_{10}(A) = 826.0938 + 0.5\cdot(-3.75) = 824.2188$, новый остаток $820 - 824.2188 = -4.2188$  
$F_{10}(B) = 878.5938 + 0.5\cdot3.75 = 880.4688$, новый остаток $880 - 880.4688 = -0.4688$  
$F_{10}(C) = 1121.4062 + 0.5\cdot(-3.75) = 1119.5312$, новый остаток $1120 - 1119.5312 = 0.4688$  
$F_{10}(D) = 1173.9062 + 0.5\cdot3.75 = 1175.7812$, новый остаток $1180 - 1175.7812 = 4.2188$

**Итог шага 10:** $\sum|r_i| = 9.375$ (было 15), $\sum r_i^2 = 36.0352$ (было 78.2227).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 3.75 осталось 1.875
— на следующем шаге эта колонка выиграет, только если 1.875 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F10, r10_new, tD = stage_D(F9, h10, LR, m=10, best=best10)
display(tD)
print(f"Σ|r| = {np.abs(r10_new).sum():g} (было {np.abs(r10).sum():g}),  Σr² = {(r10_new**2).sum():g}")
plot_step(10, F9, F10, r10, r10_new, best10, cands10)

## Шаг 11:  $F_{11} = F_{10} + \nu\,h_{11}$,  $\nu = 0.5$

**Где мы.** После шага 10 сумма модулей ошибок $\sum|r_i| = 9.375$, сумма квадратов $\sum r_i^2 = 36.0352$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 2.3438 больше вклада другой колонки.

### 11.A — Остатки текущей модели $F_{10}$

$r_i = y_i - F_{10}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 824.2188 = -4.2188$  
$r_B = 880 - 880.4688 = -0.4688$  
$r_C = 1120 - 1119.5312 = 0.4688$  
$r_D = 1180 - 1175.7812 = 4.2188$

Проверка: $\sum|r_i| = 4.2188 + 0.4688 + 0.4688 + 4.2188 = 9.375$.

In [ ]:
r11, tA = stage_A(F10, m=11)
display(tA)
print(f"Σ|r| = {np.abs(r11).sum():g},  Σr² = {(r11**2).sum():g}")

### 11.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-4.2188) + (-0.4688))/2 = -2.3438$; остаётся $((-4.2188) - (-2.3438))^2 + ((-0.4688) - (-2.3438))^2 = 7.0312$
- **экран 15"** = {C, D}: $\gamma = (0.4688 + 4.2188)/2 = 2.3438$; остаётся $(0.4688 - 2.3438)^2 + (4.2188 - 2.3438)^2 = 7.0312$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-4.2188) + 0.4688)/2 = -1.875$; остаётся $((-4.2188) - (-1.875))^2 + (0.4688 - (-1.875))^2 = 10.9863$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-0.4688) + 4.2188)/2 = 1.875$; остаётся $((-0.4688) - 1.875)^2 + (4.2188 - 1.875)^2 = 10.9863$

In [ ]:
cands11, tB = stage_B(r11)
display(tB)

### 11.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $7.0312 + 7.0312 = 14.0625$;
SSD: $10.9863 + 10.9863 = 21.9727$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{11}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best11, tC = stage_C(cands11)
display(tC)
h11 = cands11[best11]["corr"]
print("h_11 =", dict(zip(names, h11.round(4).tolist())))

### 11.D — Обновление модели с $\nu = 0.5$

$F_{11}(x_i) = F_{10}(x_i) + \nu\,h_{11}(x_i)$ — прибавляем только долю поправки.

$F_{11}(A) = 824.2188 + 0.5\cdot(-2.3438) = 823.0469$, новый остаток $820 - 823.0469 = -3.0469$  
$F_{11}(B) = 880.4688 + 0.5\cdot(-2.3438) = 879.2969$, новый остаток $880 - 879.2969 = 0.7031$  
$F_{11}(C) = 1119.5312 + 0.5\cdot2.3438 = 1120.7031$, новый остаток $1120 - 1120.7031 = -0.7031$  
$F_{11}(D) = 1175.7812 + 0.5\cdot2.3438 = 1176.9531$, новый остаток $1180 - 1176.9531 = 3.0469$

**Итог шага 11:** $\sum|r_i| = 7.5$ (было 9.375), $\sum r_i^2 = 19.5557$ (было 36.0352).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 2.3438 осталось 1.1719
— на следующем шаге эта колонка выиграет, только если 1.1719 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F11, r11_new, tD = stage_D(F10, h11, LR, m=11, best=best11)
display(tD)
print(f"Σ|r| = {np.abs(r11_new).sum():g} (было {np.abs(r11).sum():g}),  Σr² = {(r11_new**2).sum():g}")
plot_step(11, F10, F11, r11, r11_new, best11, cands11)

## Шаг 12:  $F_{12} = F_{11} + \nu\,h_{12}$,  $\nu = 0.5$

**Где мы.** После шага 11 сумма модулей ошибок $\sum|r_i| = 7.5$, сумма квадратов $\sum r_i^2 = 19.5557$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 1.875 больше вклада другой колонки.

### 12.A — Остатки текущей модели $F_{11}$

$r_i = y_i - F_{11}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 823.0469 = -3.0469$  
$r_B = 880 - 879.2969 = 0.7031$  
$r_C = 1120 - 1120.7031 = -0.7031$  
$r_D = 1180 - 1176.9531 = 3.0469$

Проверка: $\sum|r_i| = 3.0469 + 0.7031 + 0.7031 + 3.0469 = 7.5$.

In [ ]:
r12, tA = stage_A(F11, m=12)
display(tA)
print(f"Σ|r| = {np.abs(r12).sum():g},  Σr² = {(r12**2).sum():g}")

### 12.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-3.0469) + 0.7031)/2 = -1.1719$; остаётся $((-3.0469) - (-1.1719))^2 + (0.7031 - (-1.1719))^2 = 7.0312$
- **экран 15"** = {C, D}: $\gamma = ((-0.7031) + 3.0469)/2 = 1.1719$; остаётся $((-0.7031) - 1.1719)^2 + (3.0469 - 1.1719)^2 = 7.0312$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-3.0469) + (-0.7031))/2 = -1.875$; остаётся $((-3.0469) - (-1.875))^2 + ((-0.7031) - (-1.875))^2 = 2.7466$
- **ssd 512 ГБ** = {B, D}: $\gamma = (0.7031 + 3.0469)/2 = 1.875$; остаётся $(0.7031 - 1.875)^2 + (3.0469 - 1.875)^2 = 2.7466$

In [ ]:
cands12, tB = stage_B(r12)
display(tB)

### 12.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $7.0312 + 7.0312 = 14.0625$;
SSD: $2.7466 + 2.7466 = 5.4932$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{12}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best12, tC = stage_C(cands12)
display(tC)
h12 = cands12[best12]["corr"]
print("h_12 =", dict(zip(names, h12.round(4).tolist())))

### 12.D — Обновление модели с $\nu = 0.5$

$F_{12}(x_i) = F_{11}(x_i) + \nu\,h_{12}(x_i)$ — прибавляем только долю поправки.

$F_{12}(A) = 823.0469 + 0.5\cdot(-1.875) = 822.1094$, новый остаток $820 - 822.1094 = -2.1094$  
$F_{12}(B) = 879.2969 + 0.5\cdot1.875 = 880.2344$, новый остаток $880 - 880.2344 = -0.2344$  
$F_{12}(C) = 1120.7031 + 0.5\cdot(-1.875) = 1119.7656$, новый остаток $1120 - 1119.7656 = 0.2344$  
$F_{12}(D) = 1176.9531 + 0.5\cdot1.875 = 1177.8906$, новый остаток $1180 - 1177.8906 = 2.1094$

**Итог шага 12:** $\sum|r_i| = 4.6875$ (было 7.5), $\sum r_i^2 = 9.0088$ (было 19.5557).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 1.875 осталось 0.9375
— на следующем шаге эта колонка выиграет, только если 0.9375 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F12, r12_new, tD = stage_D(F11, h12, LR, m=12, best=best12)
display(tD)
print(f"Σ|r| = {np.abs(r12_new).sum():g} (было {np.abs(r12).sum():g}),  Σr² = {(r12_new**2).sum():g}")
plot_step(12, F11, F12, r12, r12_new, best12, cands12)

## Шаг 13:  $F_{13} = F_{12} + \nu\,h_{13}$,  $\nu = 0.5$

**Где мы.** После шага 12 сумма модулей ошибок $\sum|r_i| = 4.6875$, сумма квадратов $\sum r_i^2 = 9.0088$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 1.1719 больше вклада другой колонки.

### 13.A — Остатки текущей модели $F_{12}$

$r_i = y_i - F_{12}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 822.1094 = -2.1094$  
$r_B = 880 - 880.2344 = -0.2344$  
$r_C = 1120 - 1119.7656 = 0.2344$  
$r_D = 1180 - 1177.8906 = 2.1094$

Проверка: $\sum|r_i| = 2.1094 + 0.2344 + 0.2344 + 2.1094 = 4.6875$.

In [ ]:
r13, tA = stage_A(F12, m=13)
display(tA)
print(f"Σ|r| = {np.abs(r13).sum():g},  Σr² = {(r13**2).sum():g}")

### 13.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-2.1094) + (-0.2344))/2 = -1.1719$; остаётся $((-2.1094) - (-1.1719))^2 + ((-0.2344) - (-1.1719))^2 = 1.7578$
- **экран 15"** = {C, D}: $\gamma = (0.2344 + 2.1094)/2 = 1.1719$; остаётся $(0.2344 - 1.1719)^2 + (2.1094 - 1.1719)^2 = 1.7578$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-2.1094) + 0.2344)/2 = -0.9375$; остаётся $((-2.1094) - (-0.9375))^2 + (0.2344 - (-0.9375))^2 = 2.7466$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-0.2344) + 2.1094)/2 = 0.9375$; остаётся $((-0.2344) - 0.9375)^2 + (2.1094 - 0.9375)^2 = 2.7466$

In [ ]:
cands13, tB = stage_B(r13)
display(tB)

### 13.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $1.7578 + 1.7578 = 3.5156$;
SSD: $2.7466 + 2.7466 = 5.4932$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{13}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best13, tC = stage_C(cands13)
display(tC)
h13 = cands13[best13]["corr"]
print("h_13 =", dict(zip(names, h13.round(4).tolist())))

### 13.D — Обновление модели с $\nu = 0.5$

$F_{13}(x_i) = F_{12}(x_i) + \nu\,h_{13}(x_i)$ — прибавляем только долю поправки.

$F_{13}(A) = 822.1094 + 0.5\cdot(-1.1719) = 821.5234$, новый остаток $820 - 821.5234 = -1.5234$  
$F_{13}(B) = 880.2344 + 0.5\cdot(-1.1719) = 879.6484$, новый остаток $880 - 879.6484 = 0.3516$  
$F_{13}(C) = 1119.7656 + 0.5\cdot1.1719 = 1120.3516$, новый остаток $1120 - 1120.3516 = -0.3516$  
$F_{13}(D) = 1177.8906 + 0.5\cdot1.1719 = 1178.4766$, новый остаток $1180 - 1178.4766 = 1.5234$

**Итог шага 13:** $\sum|r_i| = 3.75$ (было 4.6875), $\sum r_i^2 = 4.8889$ (было 9.0088).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 1.1719 осталось 0.5859
— на следующем шаге эта колонка выиграет, только если 0.5859 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F13, r13_new, tD = stage_D(F12, h13, LR, m=13, best=best13)
display(tD)
print(f"Σ|r| = {np.abs(r13_new).sum():g} (было {np.abs(r13).sum():g}),  Σr² = {(r13_new**2).sum():g}")
plot_step(13, F12, F13, r13, r13_new, best13, cands13)

## Шаг 14:  $F_{14} = F_{13} + \nu\,h_{14}$,  $\nu = 0.5$

**Где мы.** После шага 13 сумма модулей ошибок $\sum|r_i| = 3.75$, сумма квадратов $\sum r_i^2 = 4.8889$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 0.9375 больше вклада другой колонки.

### 14.A — Остатки текущей модели $F_{13}$

$r_i = y_i - F_{13}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 821.5234 = -1.5234$  
$r_B = 880 - 879.6484 = 0.3516$  
$r_C = 1120 - 1120.3516 = -0.3516$  
$r_D = 1180 - 1178.4766 = 1.5234$

Проверка: $\sum|r_i| = 1.5234 + 0.3516 + 0.3516 + 1.5234 = 3.75$.

In [ ]:
r14, tA = stage_A(F13, m=14)
display(tA)
print(f"Σ|r| = {np.abs(r14).sum():g},  Σr² = {(r14**2).sum():g}")

### 14.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-1.5234) + 0.3516)/2 = -0.5859$; остаётся $((-1.5234) - (-0.5859))^2 + (0.3516 - (-0.5859))^2 = 1.7578$
- **экран 15"** = {C, D}: $\gamma = ((-0.3516) + 1.5234)/2 = 0.5859$; остаётся $((-0.3516) - 0.5859)^2 + (1.5234 - 0.5859)^2 = 1.7578$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-1.5234) + (-0.3516))/2 = -0.9375$; остаётся $((-1.5234) - (-0.9375))^2 + ((-0.3516) - (-0.9375))^2 = 0.6866$
- **ssd 512 ГБ** = {B, D}: $\gamma = (0.3516 + 1.5234)/2 = 0.9375$; остаётся $(0.3516 - 0.9375)^2 + (1.5234 - 0.9375)^2 = 0.6866$

In [ ]:
cands14, tB = stage_B(r14)
display(tB)

### 14.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $1.7578 + 1.7578 = 3.5156$;
SSD: $0.6866 + 0.6866 = 1.3733$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{14}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best14, tC = stage_C(cands14)
display(tC)
h14 = cands14[best14]["corr"]
print("h_14 =", dict(zip(names, h14.round(4).tolist())))

### 14.D — Обновление модели с $\nu = 0.5$

$F_{14}(x_i) = F_{13}(x_i) + \nu\,h_{14}(x_i)$ — прибавляем только долю поправки.

$F_{14}(A) = 821.5234 + 0.5\cdot(-0.9375) = 821.0547$, новый остаток $820 - 821.0547 = -1.0547$  
$F_{14}(B) = 879.6484 + 0.5\cdot0.9375 = 880.1172$, новый остаток $880 - 880.1172 = -0.1172$  
$F_{14}(C) = 1120.3516 + 0.5\cdot(-0.9375) = 1119.8828$, новый остаток $1120 - 1119.8828 = 0.1172$  
$F_{14}(D) = 1178.4766 + 0.5\cdot0.9375 = 1178.9453$, новый остаток $1180 - 1178.9453 = 1.0547$

**Итог шага 14:** $\sum|r_i| = 2.3438$ (было 3.75), $\sum r_i^2 = 2.2522$ (было 4.8889).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 0.9375 осталось 0.4688
— на следующем шаге эта колонка выиграет, только если 0.4688 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F14, r14_new, tD = stage_D(F13, h14, LR, m=14, best=best14)
display(tD)
print(f"Σ|r| = {np.abs(r14_new).sum():g} (было {np.abs(r14).sum():g}),  Σr² = {(r14_new**2).sum():g}")
plot_step(14, F13, F14, r14, r14_new, best14, cands14)

## Шаг 15:  $F_{15} = F_{14} + \nu\,h_{15}$,  $\nu = 0.5$

**Где мы.** После шага 14 сумма модулей ошибок $\sum|r_i| = 2.3438$, сумма квадратов $\sum r_i^2 = 2.2522$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 0.5859 больше вклада другой колонки.

### 15.A — Остатки текущей модели $F_{14}$

$r_i = y_i - F_{14}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 821.0547 = -1.0547$  
$r_B = 880 - 880.1172 = -0.1172$  
$r_C = 1120 - 1119.8828 = 0.1172$  
$r_D = 1180 - 1178.9453 = 1.0547$

Проверка: $\sum|r_i| = 1.0547 + 0.1172 + 0.1172 + 1.0547 = 2.3438$.

In [ ]:
r15, tA = stage_A(F14, m=15)
display(tA)
print(f"Σ|r| = {np.abs(r15).sum():g},  Σr² = {(r15**2).sum():g}")

### 15.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-1.0547) + (-0.1172))/2 = -0.5859$; остаётся $((-1.0547) - (-0.5859))^2 + ((-0.1172) - (-0.5859))^2 = 0.4395$
- **экран 15"** = {C, D}: $\gamma = (0.1172 + 1.0547)/2 = 0.5859$; остаётся $(0.1172 - 0.5859)^2 + (1.0547 - 0.5859)^2 = 0.4395$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-1.0547) + 0.1172)/2 = -0.4688$; остаётся $((-1.0547) - (-0.4688))^2 + (0.1172 - (-0.4688))^2 = 0.6866$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-0.1172) + 1.0547)/2 = 0.4688$; остаётся $((-0.1172) - 0.4688)^2 + (1.0547 - 0.4688)^2 = 0.6866$

In [ ]:
cands15, tB = stage_B(r15)
display(tB)

### 15.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $0.4395 + 0.4395 = 0.8789$;
SSD: $0.6866 + 0.6866 = 1.3733$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{15}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best15, tC = stage_C(cands15)
display(tC)
h15 = cands15[best15]["corr"]
print("h_15 =", dict(zip(names, h15.round(4).tolist())))

### 15.D — Обновление модели с $\nu = 0.5$

$F_{15}(x_i) = F_{14}(x_i) + \nu\,h_{15}(x_i)$ — прибавляем только долю поправки.

$F_{15}(A) = 821.0547 + 0.5\cdot(-0.5859) = 820.7617$, новый остаток $820 - 820.7617 = -0.7617$  
$F_{15}(B) = 880.1172 + 0.5\cdot(-0.5859) = 879.8242$, новый остаток $880 - 879.8242 = 0.1758$  
$F_{15}(C) = 1119.8828 + 0.5\cdot0.5859 = 1120.1758$, новый остаток $1120 - 1120.1758 = -0.1758$  
$F_{15}(D) = 1178.9453 + 0.5\cdot0.5859 = 1179.2383$, новый остаток $1180 - 1179.2383 = 0.7617$

**Итог шага 15:** $\sum|r_i| = 1.875$ (было 2.3438), $\sum r_i^2 = 1.2222$ (было 2.2522).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 0.5859 осталось 0.293
— на следующем шаге эта колонка выиграет, только если 0.293 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F15, r15_new, tD = stage_D(F14, h15, LR, m=15, best=best15)
display(tD)
print(f"Σ|r| = {np.abs(r15_new).sum():g} (было {np.abs(r15).sum():g}),  Σr² = {(r15_new**2).sum():g}")
plot_step(15, F14, F15, r15, r15_new, best15, cands15)

## Шаг 16:  $F_{16} = F_{15} + \nu\,h_{16}$,  $\nu = 0.5$

**Где мы.** После шага 15 сумма модулей ошибок $\sum|r_i| = 1.875$, сумма квадратов $\sum r_i^2 = 1.2222$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 0.4688 больше вклада другой колонки.

### 16.A — Остатки текущей модели $F_{15}$

$r_i = y_i - F_{15}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 820.7617 = -0.7617$  
$r_B = 880 - 879.8242 = 0.1758$  
$r_C = 1120 - 1120.1758 = -0.1758$  
$r_D = 1180 - 1179.2383 = 0.7617$

Проверка: $\sum|r_i| = 0.7617 + 0.1758 + 0.1758 + 0.7617 = 1.875$.

In [ ]:
r16, tA = stage_A(F15, m=16)
display(tA)
print(f"Σ|r| = {np.abs(r16).sum():g},  Σr² = {(r16**2).sum():g}")

### 16.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-0.7617) + 0.1758)/2 = -0.293$; остаётся $((-0.7617) - (-0.293))^2 + (0.1758 - (-0.293))^2 = 0.4395$
- **экран 15"** = {C, D}: $\gamma = ((-0.1758) + 0.7617)/2 = 0.293$; остаётся $((-0.1758) - 0.293)^2 + (0.7617 - 0.293)^2 = 0.4395$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-0.7617) + (-0.1758))/2 = -0.4688$; остаётся $((-0.7617) - (-0.4688))^2 + ((-0.1758) - (-0.4688))^2 = 0.1717$
- **ssd 512 ГБ** = {B, D}: $\gamma = (0.1758 + 0.7617)/2 = 0.4688$; остаётся $(0.1758 - 0.4688)^2 + (0.7617 - 0.4688)^2 = 0.1717$

In [ ]:
cands16, tB = stage_B(r16)
display(tB)

### 16.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $0.4395 + 0.4395 = 0.8789$;
SSD: $0.1717 + 0.1717 = 0.3433$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{16}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best16, tC = stage_C(cands16)
display(tC)
h16 = cands16[best16]["corr"]
print("h_16 =", dict(zip(names, h16.round(4).tolist())))

### 16.D — Обновление модели с $\nu = 0.5$

$F_{16}(x_i) = F_{15}(x_i) + \nu\,h_{16}(x_i)$ — прибавляем только долю поправки.

$F_{16}(A) = 820.7617 + 0.5\cdot(-0.4688) = 820.5273$, новый остаток $820 - 820.5273 = -0.5273$  
$F_{16}(B) = 879.8242 + 0.5\cdot0.4688 = 880.0586$, новый остаток $880 - 880.0586 = -0.0586$  
$F_{16}(C) = 1120.1758 + 0.5\cdot(-0.4688) = 1119.9414$, новый остаток $1120 - 1119.9414 = 0.0586$  
$F_{16}(D) = 1179.2383 + 0.5\cdot0.4688 = 1179.4727$, новый остаток $1180 - 1179.4727 = 0.5273$

**Итог шага 16:** $\sum|r_i| = 1.1719$ (было 1.875), $\sum r_i^2 = 0.563$ (было 1.2222).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 0.4688 осталось 0.2344
— на следующем шаге эта колонка выиграет, только если 0.2344 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F16, r16_new, tD = stage_D(F15, h16, LR, m=16, best=best16)
display(tD)
print(f"Σ|r| = {np.abs(r16_new).sum():g} (было {np.abs(r16).sum():g}),  Σr² = {(r16_new**2).sum():g}")
plot_step(16, F15, F16, r16, r16_new, best16, cands16)

## Шаг 17:  $F_{17} = F_{16} + \nu\,h_{17}$,  $\nu = 0.5$

**Где мы.** После шага 16 сумма модулей ошибок $\sum|r_i| = 1.1719$, сумма квадратов $\sum r_i^2 = 0.563$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 0.293 больше вклада другой колонки.

### 17.A — Остатки текущей модели $F_{16}$

$r_i = y_i - F_{16}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 820.5273 = -0.5273$  
$r_B = 880 - 880.0586 = -0.0586$  
$r_C = 1120 - 1119.9414 = 0.0586$  
$r_D = 1180 - 1179.4727 = 0.5273$

Проверка: $\sum|r_i| = 0.5273 + 0.0586 + 0.0586 + 0.5273 = 1.1719$.

In [ ]:
r17, tA = stage_A(F16, m=17)
display(tA)
print(f"Σ|r| = {np.abs(r17).sum():g},  Σr² = {(r17**2).sum():g}")

### 17.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-0.5273) + (-0.0586))/2 = -0.293$; остаётся $((-0.5273) - (-0.293))^2 + ((-0.0586) - (-0.293))^2 = 0.1099$
- **экран 15"** = {C, D}: $\gamma = (0.0586 + 0.5273)/2 = 0.293$; остаётся $(0.0586 - 0.293)^2 + (0.5273 - 0.293)^2 = 0.1099$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-0.5273) + 0.0586)/2 = -0.2344$; остаётся $((-0.5273) - (-0.2344))^2 + (0.0586 - (-0.2344))^2 = 0.1717$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-0.0586) + 0.5273)/2 = 0.2344$; остаётся $((-0.0586) - 0.2344)^2 + (0.5273 - 0.2344)^2 = 0.1717$

In [ ]:
cands17, tB = stage_B(r17)
display(tB)

### 17.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $0.1099 + 0.1099 = 0.2197$;
SSD: $0.1717 + 0.1717 = 0.3433$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{17}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best17, tC = stage_C(cands17)
display(tC)
h17 = cands17[best17]["corr"]
print("h_17 =", dict(zip(names, h17.round(4).tolist())))

### 17.D — Обновление модели с $\nu = 0.5$

$F_{17}(x_i) = F_{16}(x_i) + \nu\,h_{17}(x_i)$ — прибавляем только долю поправки.

$F_{17}(A) = 820.5273 + 0.5\cdot(-0.293) = 820.3809$, новый остаток $820 - 820.3809 = -0.3809$  
$F_{17}(B) = 880.0586 + 0.5\cdot(-0.293) = 879.9121$, новый остаток $880 - 879.9121 = 0.0879$  
$F_{17}(C) = 1119.9414 + 0.5\cdot0.293 = 1120.0879$, новый остаток $1120 - 1120.0879 = -0.0879$  
$F_{17}(D) = 1179.4727 + 0.5\cdot0.293 = 1179.6191$, новый остаток $1180 - 1179.6191 = 0.3809$

**Итог шага 17:** $\sum|r_i| = 0.9375$ (было 1.1719), $\sum r_i^2 = 0.3056$ (было 0.563).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 0.293 осталось 0.1465
— на следующем шаге эта колонка выиграет, только если 0.1465 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F17, r17_new, tD = stage_D(F16, h17, LR, m=17, best=best17)
display(tD)
print(f"Σ|r| = {np.abs(r17_new).sum():g} (было {np.abs(r17).sum():g}),  Σr² = {(r17_new**2).sum():g}")
plot_step(17, F16, F17, r17, r17_new, best17, cands17)

## Шаг 18:  $F_{18} = F_{17} + \nu\,h_{18}$,  $\nu = 0.5$

**Где мы.** После шага 17 сумма модулей ошибок $\sum|r_i| = 0.9375$, сумма квадратов $\sum r_i^2 = 0.3056$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 0.2344 больше вклада другой колонки.

### 18.A — Остатки текущей модели $F_{17}$

$r_i = y_i - F_{17}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 820.3809 = -0.3809$  
$r_B = 880 - 879.9121 = 0.0879$  
$r_C = 1120 - 1120.0879 = -0.0879$  
$r_D = 1180 - 1179.6191 = 0.3809$

Проверка: $\sum|r_i| = 0.3809 + 0.0879 + 0.0879 + 0.3809 = 0.9375$.

In [ ]:
r18, tA = stage_A(F17, m=18)
display(tA)
print(f"Σ|r| = {np.abs(r18).sum():g},  Σr² = {(r18**2).sum():g}")

### 18.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-0.3809) + 0.0879)/2 = -0.1465$; остаётся $((-0.3809) - (-0.1465))^2 + (0.0879 - (-0.1465))^2 = 0.1099$
- **экран 15"** = {C, D}: $\gamma = ((-0.0879) + 0.3809)/2 = 0.1465$; остаётся $((-0.0879) - 0.1465)^2 + (0.3809 - 0.1465)^2 = 0.1099$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-0.3809) + (-0.0879))/2 = -0.2344$; остаётся $((-0.3809) - (-0.2344))^2 + ((-0.0879) - (-0.2344))^2 = 0.0429$
- **ssd 512 ГБ** = {B, D}: $\gamma = (0.0879 + 0.3809)/2 = 0.2344$; остаётся $(0.0879 - 0.2344)^2 + (0.3809 - 0.2344)^2 = 0.0429$

In [ ]:
cands18, tB = stage_B(r18)
display(tB)

### 18.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $0.1099 + 0.1099 = 0.2197$;
SSD: $0.0429 + 0.0429 = 0.0858$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{18}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best18, tC = stage_C(cands18)
display(tC)
h18 = cands18[best18]["corr"]
print("h_18 =", dict(zip(names, h18.round(4).tolist())))

### 18.D — Обновление модели с $\nu = 0.5$

$F_{18}(x_i) = F_{17}(x_i) + \nu\,h_{18}(x_i)$ — прибавляем только долю поправки.

$F_{18}(A) = 820.3809 + 0.5\cdot(-0.2344) = 820.2637$, новый остаток $820 - 820.2637 = -0.2637$  
$F_{18}(B) = 879.9121 + 0.5\cdot0.2344 = 880.0293$, новый остаток $880 - 880.0293 = -0.0293$  
$F_{18}(C) = 1120.0879 + 0.5\cdot(-0.2344) = 1119.9707$, новый остаток $1120 - 1119.9707 = 0.0293$  
$F_{18}(D) = 1179.6191 + 0.5\cdot0.2344 = 1179.7363$, новый остаток $1180 - 1179.7363 = 0.2637$

**Итог шага 18:** $\sum|r_i| = 0.5859$ (было 0.9375), $\sum r_i^2 = 0.1408$ (было 0.3056).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 0.2344 осталось 0.1172
— на следующем шаге эта колонка выиграет, только если 0.1172 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F18, r18_new, tD = stage_D(F17, h18, LR, m=18, best=best18)
display(tD)
print(f"Σ|r| = {np.abs(r18_new).sum():g} (было {np.abs(r18).sum():g}),  Σr² = {(r18_new**2).sum():g}")
plot_step(18, F17, F18, r18, r18_new, best18, cands18)

## Шаг 19:  $F_{19} = F_{18} + \nu\,h_{19}$,  $\nu = 0.5$

**Где мы.** После шага 18 сумма модулей ошибок $\sum|r_i| = 0.5859$, сумма квадратов $\sum r_i^2 = 0.1408$.
На этом шаге модель выберет колонку **«экран»** — её невыученный вклад 0.1465 больше вклада другой колонки.

### 19.A — Остатки текущей модели $F_{18}$

$r_i = y_i - F_{18}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 820.2637 = -0.2637$  
$r_B = 880 - 880.0293 = -0.0293$  
$r_C = 1120 - 1119.9707 = 0.0293$  
$r_D = 1180 - 1179.7363 = 0.2637$

Проверка: $\sum|r_i| = 0.2637 + 0.0293 + 0.0293 + 0.2637 = 0.5859$.

In [ ]:
r19, tA = stage_A(F18, m=19)
display(tA)
print(f"Σ|r| = {np.abs(r19).sum():g},  Σr² = {(r19**2).sum():g}")

### 19.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-0.2637) + (-0.0293))/2 = -0.1465$; остаётся $((-0.2637) - (-0.1465))^2 + ((-0.0293) - (-0.1465))^2 = 0.0275$
- **экран 15"** = {C, D}: $\gamma = (0.0293 + 0.2637)/2 = 0.1465$; остаётся $(0.0293 - 0.1465)^2 + (0.2637 - 0.1465)^2 = 0.0275$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-0.2637) + 0.0293)/2 = -0.1172$; остаётся $((-0.2637) - (-0.1172))^2 + (0.0293 - (-0.1172))^2 = 0.0429$
- **ssd 512 ГБ** = {B, D}: $\gamma = ((-0.0293) + 0.2637)/2 = 0.1172$; остаётся $((-0.0293) - 0.1172)^2 + (0.2637 - 0.1172)^2 = 0.0429$

In [ ]:
cands19, tB = stage_B(r19)
display(tB)

### 19.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $0.0275 + 0.0275 = 0.0549$;
SSD: $0.0429 + 0.0429 = 0.0858$.

Минимум у **«экран»** — в 1.56 раза меньше. Значит $h_{19}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best19, tC = stage_C(cands19)
display(tC)
h19 = cands19[best19]["corr"]
print("h_19 =", dict(zip(names, h19.round(4).tolist())))

### 19.D — Обновление модели с $\nu = 0.5$

$F_{19}(x_i) = F_{18}(x_i) + \nu\,h_{19}(x_i)$ — прибавляем только долю поправки.

$F_{19}(A) = 820.2637 + 0.5\cdot(-0.1465) = 820.1904$, новый остаток $820 - 820.1904 = -0.1904$  
$F_{19}(B) = 880.0293 + 0.5\cdot(-0.1465) = 879.9561$, новый остаток $880 - 879.9561 = 0.0439$  
$F_{19}(C) = 1119.9707 + 0.5\cdot0.1465 = 1120.0439$, новый остаток $1120 - 1120.0439 = -0.0439$  
$F_{19}(D) = 1179.7363 + 0.5\cdot0.1465 = 1179.8096$, новый остаток $1180 - 1179.8096 = 0.1904$

**Итог шага 19:** $\sum|r_i| = 0.4688$ (было 0.5859), $\sum r_i^2 = 0.0764$ (было 0.1408).
Невыученная часть вклада колонки «экран» умножилась на $(1-\nu) = 0.5$: из 0.1465 осталось 0.0732
— на следующем шаге эта колонка выиграет, только если 0.0732 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F19, r19_new, tD = stage_D(F18, h19, LR, m=19, best=best19)
display(tD)
print(f"Σ|r| = {np.abs(r19_new).sum():g} (было {np.abs(r19).sum():g}),  Σr² = {(r19_new**2).sum():g}")
plot_step(19, F18, F19, r19, r19_new, best19, cands19)

## Шаг 20:  $F_{20} = F_{19} + \nu\,h_{20}$,  $\nu = 0.5$

**Где мы.** После шага 19 сумма модулей ошибок $\sum|r_i| = 0.4688$, сумма квадратов $\sum r_i^2 = 0.0764$.
На этом шаге модель выберет колонку **«ssd»** — её невыученный вклад 0.1172 больше вклада другой колонки.

### 20.A — Остатки текущей модели $F_{19}$

$r_i = y_i - F_{19}(x_i)$ — антиградиент квадратичной потери: $-\partial L/\partial F = y - F$. Это «что модель ещё не выучила»; минус — завысили, плюс — занизили.

$r_A = 820 - 820.1904 = -0.1904$  
$r_B = 880 - 879.9561 = 0.0439$  
$r_C = 1120 - 1120.0439 = -0.0439$  
$r_D = 1180 - 1179.8096 = 0.1904$

Проверка: $\sum|r_i| = 0.1904 + 0.0439 + 0.0439 + 0.1904 = 0.4688$.

In [ ]:
r20, tA = stage_A(F19, m=20)
display(tA)
print(f"Σ|r| = {np.abs(r20).sum():g},  Σr² = {(r20**2).sum():g}")

### 20.B — Группировка по колонкам и поправки групп

Каждая колонка делит объекты на две группы, группа получает одно число $\gamma_G$. Лучшее такое число для квадратичной потери — среднее остатков группы:
$\gamma_G = \arg\min_\gamma \sum_{i\in G}(r_i-\gamma)^2 = \frac{1}{|G|}\sum_{i\in G} r_i$.
Группируем по признаку, а не по остаткам: признак известен у любого нового объекта, остаток — нет.

- **экран 13"** = {A, B}: $\gamma = ((-0.1904) + 0.0439)/2 = -0.0732$; остаётся $((-0.1904) - (-0.0732))^2 + (0.0439 - (-0.0732))^2 = 0.0275$
- **экран 15"** = {C, D}: $\gamma = ((-0.0439) + 0.1904)/2 = 0.0732$; остаётся $((-0.0439) - 0.0732)^2 + (0.1904 - 0.0732)^2 = 0.0275$
- **ssd 256 ГБ** = {A, C}: $\gamma = ((-0.1904) + (-0.0439))/2 = -0.1172$; остаётся $((-0.1904) - (-0.1172))^2 + ((-0.0439) - (-0.1172))^2 = 0.0107$
- **ssd 512 ГБ** = {B, D}: $\gamma = (0.0439 + 0.1904)/2 = 0.1172$; остаётся $(0.0439 - 0.1172)^2 + (0.1904 - 0.1172)^2 = 0.0107$

In [ ]:
cands20, tB = stage_B(r20)
display(tB)

### 20.C — Выбор колонки

Складываем ошибку, оставшуюся в обеих группах каждой колонки, и берём минимум:
экран: $0.0275 + 0.0275 = 0.0549$;
SSD: $0.0107 + 0.0107 = 0.0215$.

Минимум у **«ssd»** — в 2.56 раза меньше. Значит $h_{20}$ строится по этой колонке: каждому объекту достаётся $\gamma$ его группы.

In [ ]:
best20, tC = stage_C(cands20)
display(tC)
h20 = cands20[best20]["corr"]
print("h_20 =", dict(zip(names, h20.round(4).tolist())))

### 20.D — Обновление модели с $\nu = 0.5$

$F_{20}(x_i) = F_{19}(x_i) + \nu\,h_{20}(x_i)$ — прибавляем только долю поправки.

$F_{20}(A) = 820.1904 + 0.5\cdot(-0.1172) = 820.1318$, новый остаток $820 - 820.1318 = -0.1318$  
$F_{20}(B) = 879.9561 + 0.5\cdot0.1172 = 880.0146$, новый остаток $880 - 880.0146 = -0.0146$  
$F_{20}(C) = 1120.0439 + 0.5\cdot(-0.1172) = 1119.9854$, новый остаток $1120 - 1119.9854 = 0.0146$  
$F_{20}(D) = 1179.8096 + 0.5\cdot0.1172 = 1179.8682$, новый остаток $1180 - 1179.8682 = 0.1318$

**Итог шага 20:** $\sum|r_i| = 0.293$ (было 0.4688), $\sum r_i^2 = 0.0352$ (было 0.0764).
Невыученная часть вклада колонки «ssd» умножилась на $(1-\nu) = 0.5$: из 0.1172 осталось 0.0586
— на следующем шаге эта колонка выиграет, только если 0.0586 всё ещё больше невыученного вклада другой колонки.

In [ ]:
F20, r20_new, tD = stage_D(F19, h20, LR, m=20, best=best20)
display(tD)
print(f"Σ|r| = {np.abs(r20_new).sum():g} (было {np.abs(r20).sum():g}),  Σr² = {(r20_new**2).sum():g}")
plot_step(20, F19, F20, r20, r20_new, best20, cands20)

## Сводка по 20 шагам

Ни одной новой величины — только то, что уже посчитано выше, собранное в одну таблицу и три графика.

In [ ]:
hist_F = [F0, F1, F2, F3, F4, F5, F6, F7, F8, F9, F10, F11, F12, F13, F14, F15, F16, F17, F18, F19, F20]
hist_best = ['—', best1, best2, best3, best4, best5, best6, best7, best8, best9, best10, best11, best12, best13, best14, best15, best16, best17, best18, best19, best20]
hist_abs = [np.abs(y - F).sum() for F in hist_F]
hist_sq = [((y - F) ** 2).sum() for F in hist_F]
summary = pd.DataFrame({'выбрана колонка': hist_best, 'Σ|r|': np.round(hist_abs, 4), 'Σr²': np.round(hist_sq, 4),
                        **{f'F({n})': [round(F[i], 2) for F in hist_F] for i, n in enumerate(names)}})
summary.index.name = 'шаг'
summary

In [ ]:
steps = np.arange(len(hist_F)); Fs = np.array(hist_F)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
cols = [C_SCREEN, C_SSD, C_PRED, "#C0392B"]
for i, n in enumerate(names):
    axes[0].plot(steps, Fs[:, i], marker="o", ms=3, lw=1.5, color=cols[i], label=f"F(x_{n})")
    axes[0].axhline(y[i], color=C_TRUE, ls=":", lw=1)
axes[0].set_title("Предсказания сходятся к реальным ценам"); axes[0].set_xlabel("шаг m"); axes[0].legend(fontsize=8)
for m in range(1, len(hist_F)):
    axes[1].bar(m, LR * np.abs(hist_F[m] - hist_F[m-1]).max(), color=C_SCREEN if hist_best[m] == "экран" else C_SSD)
axes[1].set_title("Какая колонка выбрана и размер поправки |ν·h|"); axes[1].set_xlabel("шаг m")
axes[2].semilogy(steps, hist_abs, marker="o", ms=3, color=C_PRED, lw=2)
axes[2].set_title("Σ|rᵢ| в лог-шкале: прямая = геометрическое убывание"); axes[2].set_xlabel("шаг m")
plt.tight_layout(); plt.show()

## Что видно из двадцати шагов

1. **Шаги 1–3 — экран.** Его невыученный вклад 150 → 75 → 37.5 → 18.75 больше вклада SSD (30), и критерий C трижды выбирает экран.
2. **Шаг 4 — первое переключение.** 18.75 < 30: SSD объясняет больше, модель берёт его.
3. **Шаги 5–20 — чередование.** После каждого шага «проигравшая» колонка оказывается с большим невыученным куском, и они сменяют друг друга по очереди; обе величины убывают геометрически.
4. **Ошибка не обнуляется.** На 20-м шаге $\sum|r_i| \approx 0.29$: каждый шаг умножает её на $(1-\nu)$, и ни при каком конечном числе шагов множитель не становится нулём. Это свойство, а не недостаток — модель можно остановить в любой момент.
5. **Итоговая модель — сумма.** $F_{20}=F_0+\nu h_1+\dots+\nu h_{20}$: вклады шагов «экран» в сумме дают ±150, «SSD» — ±30. За двадцать осторожных шагов модель собрала те же две поправки, что при $\nu=1$ нашла за два.

**Что дальше:** зачем вообще нужен медленный шаг — валидация, переобучение и ранняя остановка.